In [20]:
# =============================================================================
# CELL 1 — Imports and environment verification
#
# Kernel: CFB Model (ARM)  ~/miniforge3/envs/cfb_model_arm/bin/python
# NumPyro 0.21.0 / JAX 0.10.0 confirmed working on cpu backend (Day 20).
#
# DB connection opened here and held open for the full notebook.
# Do not close until the final cell.
# =============================================================================

import numpy as np
import pandas as pd
import psycopg2
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random
from dataclasses import dataclass
from typing import Optional
import time

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print(f"NumPyro version : {numpyro.__version__}")
print(f"JAX version     : {jax.__version__}")
print(f"JAX backend     : {jax.default_backend()}")
print(f"pandas          : {pd.__version__}")
print(f"numpy           : {np.__version__}")
print()
print("DB connection established.")
print("Cell 1 complete.")

NumPyro version : 0.21.0
JAX version     : 0.10.0
JAX backend     : cpu
pandas          : 3.0.2
numpy           : 2.4.4

DB connection established.
Cell 1 complete.


In [21]:
# =============================================================================
# CELL 2 — Reproduce conference index maps and model infrastructure
#
# Prior corrections applied (model_05 prior predictive audit, Day 24):
#   r_negbinom       : Gamma(2.0, 0.1)  → Gamma(16.0, 2.0)
#                      mean=8, std=2; 2-sigma lower bound ≈ r=4
#                      Eliminates low-r tail that caused within-sample
#                      score explosions in prior predictive
#   sigma_conference : HalfNormal(3.0)  → HalfNormal(0.1)
#                      Log scale — HalfNormal(3.0) allowed exp(6)≈400x
#                      conference multipliers
#   sigma_attack     : HalfNormal(0.4)  → HalfNormal(0.1)
#   sigma_defense    : HalfNormal(0.4)  → HalfNormal(0.1)
#                      Combined team effect tail was driving log_mu explosions
#   sp_weight        : Normal(0, 1.0)   → Normal(0, 0.15)
#   b_close_game_epa : Normal(0, 0.5)   → Normal(0, 0.10)
#   b_close_game_def_epa: Normal(0, 0.5) → Normal(0, 0.10)
#   b_off_archetype  : Normal(0, 0.3)   → Normal(0, 0.15)
#   b_def_archetype  : Normal(0, 0.3)   → Normal(0, 0.15)
#   All other game-level coefs at Normal(0, 0.3/0.2) → Normal(0, 0.15)
#
# Architecture unchanged from audited model_04:
#   - recruiting_3yr_avg: conference-specific rec_weight, Sun Belt hard
#     non-positive constraint (TruncatedNormal high=0.0)
#   - Conference-scoped features use individual per-feature masks
#   - SCOPE_ELEVATION: Mountain West + Big 12 (not universal)
#   - mu.clip(1e-6) and r.clip(1e-6) both retained
#   - validate_args=False on NegativeBinomial2
# =============================================================================

# --- Conference index maps ---
CONFERENCES = [
    "ACC",               # 0
    "American Athletic", # 1
    "Big 12",            # 2
    "Big Ten",           # 3
    "Conference USA",    # 4
    "Mid-American",      # 5
    "Mountain West",     # 6
    "Pac-12",            # 7
    "SEC",               # 8
    "Sun Belt",          # 9
]
N_CONFERENCES = len(CONFERENCES)
conf_to_idx   = {c: i for i, c in enumerate(CONFERENCES)}
SUN_BELT_IDX  = conf_to_idx["Sun Belt"]

# --- Conference-scope sets ---
SCOPE_LAST3_OFF_EPA      = {"ACC", "Mid-American", "SEC"}
SCOPE_LAST3_DEF_EPA      = {"American Athletic", "Big Ten", "Conference USA",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_LAST3_PTS_SCORED   = {"ACC", "Big 12", "Big Ten", "Conference USA",
                             "Mid-American", "Mountain West"}
SCOPE_LAST3_PTS_ALLOWED  = {"American Athletic", "Big Ten", "Conference USA",
                             "Mountain West", "Pac-12", "Sun Belt"}
SCOPE_DAYS_SINCE         = {"American Athletic", "Big 12"}
SCOPE_CLOSE_PLAY_COUNT   = {"ACC", "American Athletic", "Big 12",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_ELEVATION          = {"Mountain West", "Big 12"}


def build_conf_mask(team_conferences, scope_set):
    return jnp.array(
        [1.0 if c in scope_set else 0.0 for c in team_conferences],
        dtype=jnp.float32
    )


# --- GameData dataclass ---
@dataclass
class GameData:
    # Index arrays (int32)
    team_idx  : jnp.ndarray
    opp_idx   : jnp.ndarray
    conf_idx  : jnp.ndarray

    # Home indicator
    is_home   : jnp.ndarray

    # Observed scores
    points    : Optional[jnp.ndarray]

    # Prior seeds
    sp_rating          : jnp.ndarray
    recruiting_3yr_avg : jnp.ndarray

    # Universal game-level features
    close_game_epa        : jnp.ndarray
    close_game_def_epa    : jnp.ndarray
    pregame_elo           : jnp.ndarray
    elo_sp_divergence     : jnp.ndarray
    last3_win_pct         : jnp.ndarray
    away_travel_distance  : jnp.ndarray
    away_tz_delta         : jnp.ndarray
    wind_chill            : jnp.ndarray
    rush_rate_std_downs   : jnp.ndarray
    rush_rate_pass_downs  : jnp.ndarray
    off_sack_rate_allowed : jnp.ndarray
    def_sack_rate         : jnp.ndarray

    # Archetype index arrays — int32, values 0-3
    off_archetype_idx : jnp.ndarray
    def_archetype_idx : jnp.ndarray

    # Conference-scoped features + masks (7 pairs)
    last3_off_epa_avg      : jnp.ndarray
    mask_last3_off_epa     : jnp.ndarray
    last3_def_epa_avg      : jnp.ndarray
    mask_last3_def_epa     : jnp.ndarray
    last3_pts_scored_avg   : jnp.ndarray
    mask_last3_pts_scored  : jnp.ndarray
    last3_pts_allowed_avg  : jnp.ndarray
    mask_last3_pts_allowed : jnp.ndarray
    days_since_last_game   : jnp.ndarray
    mask_days_since        : jnp.ndarray
    close_play_count_delta : jnp.ndarray
    mask_close_play_count  : jnp.ndarray
    away_elevation_delta   : jnp.ndarray
    mask_elevation         : jnp.ndarray


# --- model_cfb() ---
def model_cfb(data: GameData, N_teams: int):

    # ── League level ──────────────────────────────────────────────────────────
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    # Gamma(16.0, 2.0): mean=8, std=2; 2-sigma lower bound ≈ r=4
    # At mu=27, r=4: VMR = 1 + 27/4 = 7.75 — inside prior predictive target
    r_negbinom = numpyro.sample(
        "r_negbinom",
        dist.Gamma(16.0, 2.0),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Conference level ──────────────────────────────────────────────────────
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference",
        dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Team level — non-centered ─────────────────────────────────────────────
    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.1))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.1))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    # ── Prior seeds ───────────────────────────────────────────────────────────
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))

    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5), sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # ── Game-level coefficients ───────────────────────────────────────────────
    b_close_game_epa      = numpyro.sample("b_close_game_epa",      dist.Normal(0.0, 0.10))
    b_close_game_def_epa  = numpyro.sample("b_close_game_def_epa",  dist.Normal(0.0, 0.10))
    b_pregame_elo         = numpyro.sample("b_pregame_elo",         dist.Normal(0.0, 0.15))
    b_elo_sp_divergence   = numpyro.sample("b_elo_sp_divergence",   dist.Normal(0.0, 0.15))
    b_last3_win_pct       = numpyro.sample("b_last3_win_pct",       dist.Normal(0.0, 0.15))
    b_away_travel_distance= numpyro.sample("b_away_travel_distance",dist.Normal(0.0, 0.15))
    b_away_tz_delta       = numpyro.sample("b_away_tz_delta",       dist.Normal(0.0, 0.15))
    b_wind_chill          = numpyro.sample("b_wind_chill",          dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs = numpyro.sample("b_rush_rate_std_downs", dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs= numpyro.sample("b_rush_rate_pass_downs",dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed=numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate       = numpyro.sample("b_def_sack_rate",       dist.Normal(0.0, 0.15))

    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )

    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))

    # ── Linear predictor ──────────────────────────────────────────────────────
    log_mu = (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight          * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa      * data.close_game_epa
        + b_close_game_def_epa  * data.close_game_def_epa
        + b_pregame_elo         * data.pregame_elo
        + b_elo_sp_divergence   * data.elo_sp_divergence
        + b_last3_win_pct       * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )

    # ── Likelihood ────────────────────────────────────────────────────────────
    mu = jnp.exp(log_mu).clip(1e-6)
    r  = r_negbinom[data.conf_idx].clip(1e-6)

    numpyro.sample(
        "obs",
        dist.NegativeBinomial2(mu, r, validate_args=False),
        obs=data.points
    )


print("Conference index map:")
for name, idx in conf_to_idx.items():
    tag = "  <- Sun Belt (rec_weight non-positive)" if idx == SUN_BELT_IDX else ""
    print(f"  {idx:2d} : {name}{tag}")
print(f"\nN_CONFERENCES : {N_CONFERENCES}")
print(f"SUN_BELT_IDX  : {SUN_BELT_IDX}")
print()
print("build_conf_mask() defined.")
print("GameData dataclass defined.")
print("model_cfb() defined.")
print("  r_negbinom       : Gamma(16.0, 2.0) x N_CONFERENCES  [mean=8, std=2]")
print("  sigma_conference : HalfNormal(0.1)")
print("  sigma_attack     : HalfNormal(0.1)")
print("  sigma_defense    : HalfNormal(0.1)")
print("  sp_weight        : Normal(0.0, 0.15)")
print("  b_off_archetype  : Normal(0.0, 0.15) x 4")
print("  b_def_archetype  : Normal(0.0, 0.15) x 4")
print("  likelihood       : NegativeBinomial2(mu, r_negbinom[conf_idx])")
print("Cell 2 complete.")

Conference index map:
   0 : ACC
   1 : American Athletic
   2 : Big 12
   3 : Big Ten
   4 : Conference USA
   5 : Mid-American
   6 : Mountain West
   7 : Pac-12
   8 : SEC
   9 : Sun Belt  <- Sun Belt (rec_weight non-positive)

N_CONFERENCES : 10
SUN_BELT_IDX  : 9

build_conf_mask() defined.
GameData dataclass defined.
model_cfb() defined.
  r_negbinom       : Gamma(16.0, 2.0) x N_CONFERENCES  [mean=8, std=2]
  sigma_conference : HalfNormal(0.1)
  sigma_attack     : HalfNormal(0.1)
  sigma_defense    : HalfNormal(0.1)
  sp_weight        : Normal(0.0, 0.15)
  b_off_archetype  : Normal(0.0, 0.15) x 4
  b_def_archetype  : Normal(0.0, 0.15) x 4
  likelihood       : NegativeBinomial2(mu, r_negbinom[conf_idx])
Cell 2 complete.


In [22]:
# =============================================================================
# CELL 3 — Load training data from int.int_game_model_features
#
# Season filter: IN (2022, 2023, 2024). 2025 is holdout — never touched.
# FBS integrity check mandatory after load.
# All Decimal columns cast to float64 immediately.
# =============================================================================

load_sql = """
    SELECT *
    FROM int.int_game_model_features
    WHERE season IN (2022, 2023, 2024)
    ORDER BY season, game_id, team_name
"""

cur.execute(load_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df = pd.DataFrame(rows, columns=cols)

numeric_cols = [c for c in df.columns if c not in
                ['team_name', 'opponent', 'conference']]
df[numeric_cols] = df[numeric_cols].astype(float)

# --- FBS integrity check ---
print("Conference distribution:")
print(df['conference'].value_counts().sort_index().to_string())
print()

assert 'FBS Independents' not in df['conference'].values, \
    "FBS Independents found in conference — fix before proceeding"
assert 2025 not in df['season'].values, \
    "2025 holdout found in training data — fix before proceeding"
assert df['game_id'].isna().sum() == 0, \
    "Null game_id found"

# --- Shape and basic sanity ---
print(f"Rows loaded        : {len(df):,}")
print(f"Columns            : {len(df.columns)}")
print(f"Unique games       : {df['game_id'].nunique():,}")
print(f"Unique teams       : {df['team_name'].nunique():,}")
print(f"Seasons            : {sorted(df['season'].unique().tolist())}")
print(f"Nulls anywhere     : {df.isna().sum().sum()}")
print(f"points_scored mean : {df['points_scored'].mean():.1f}")
print()

assert len(df) == 3214,          f"Expected 3,214 rows, got {len(df):,}"
assert df['game_id'].nunique() == 1607, \
    f"Expected 1,607 games, got {df['game_id'].nunique():,}"
assert df.isna().sum().sum() == 0, "Nulls found in training data"

print("All load checks passed.")
print("Cell 3 complete.")

Conference distribution:
conference
ACC                  364
American Athletic    320
Big 12               366
Big Ten              420
Conference USA       246
Mid-American         294
Mountain West        282
Pac-12               222
SEC                  358
Sun Belt             342

Rows loaded        : 3,214
Columns            : 31
Unique games       : 1,607
Unique teams       : 131
Seasons            : [2022.0, 2023.0, 2024.0]
Nulls anywhere     : 0
points_scored mean : 26.9

All load checks passed.
Cell 3 complete.


In [23]:
# =============================================================================
# CELL 4 — Build index arrays and conference masks
#
# team_idx and opp_idx: 0-based integer index over unique team_name values.
# Uniqueness key: team_name (team×season pooled — 131 unique teams across
# 2022-2024; no separate index per season per locked decision).
#
# conf_idx: maps each row's conference string to conf_to_idx integer.
# Must use the exact same conf_to_idx from Cell 2.
#
# opp_idx: derived from the opponent column in int_game_model_features,
# which carries the opponent team_name for each row.
#
# All index arrays must satisfy:
#   team_idx in [0, N_teams)
#   opp_idx  in [0, N_teams)
#   conf_idx in [0, N_CONFERENCES)
# =============================================================================

# --- Team index map ---
unique_teams = sorted(df['team_name'].unique())
N_teams      = len(unique_teams)
team_to_idx  = {name: i for i, name in enumerate(unique_teams)}

print(f"N_teams : {N_teams}")

# --- Verify all opponents are in the team map ---
unknown_opps = set(df['opponent'].unique()) - set(team_to_idx.keys())
assert len(unknown_opps) == 0, \
    f"Opponents not in team_to_idx: {unknown_opps}"

# --- Build index arrays ---
team_idx = jnp.array(df['team_name'].map(team_to_idx).values, dtype=jnp.int32)
opp_idx  = jnp.array(df['opponent'].map(team_to_idx).values,  dtype=jnp.int32)
conf_idx = jnp.array(df['conference'].map(conf_to_idx).values, dtype=jnp.int32)

# --- Verify conference mapping was complete ---
assert df['conference'].map(conf_to_idx).isna().sum() == 0, \
    f"Unmapped conferences: {set(df['conference'].unique()) - set(conf_to_idx.keys())}"

# --- Range assertions ---
assert int(team_idx.min()) >= 0 and int(team_idx.max()) < N_teams, \
    f"team_idx out of range: [{int(team_idx.min())}, {int(team_idx.max())}]"
assert int(opp_idx.min()) >= 0 and int(opp_idx.max()) < N_teams, \
    f"opp_idx out of range: [{int(opp_idx.min())}, {int(opp_idx.max())}]"
assert int(conf_idx.min()) >= 0 and int(conf_idx.max()) < N_CONFERENCES, \
    f"conf_idx out of range: [{int(conf_idx.min())}, {int(conf_idx.max())}]"

print(f"team_idx : shape={team_idx.shape}  min={int(team_idx.min())}  max={int(team_idx.max())}")
print(f"opp_idx  : shape={opp_idx.shape}  min={int(opp_idx.min())}  max={int(opp_idx.max())}")
print(f"conf_idx : shape={conf_idx.shape}  min={int(conf_idx.min())}  max={int(conf_idx.max())}")

# --- Build conference masks ---
team_conferences = df['conference'].tolist()

mask_last3_off_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_OFF_EPA)
mask_last3_def_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_DEF_EPA)
mask_last3_pts_scored  = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_SCORED)
mask_last3_pts_allowed = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_ALLOWED)
mask_days_since        = build_conf_mask(team_conferences, SCOPE_DAYS_SINCE)
mask_close_play_count  = build_conf_mask(team_conferences, SCOPE_CLOSE_PLAY_COUNT)
mask_elevation         = build_conf_mask(team_conferences, SCOPE_ELEVATION)

print()
print("Conference mask non-zero counts (of 3,214 rows):")
print(f"  mask_last3_off_epa     : {int(mask_last3_off_epa.sum()):,}")
print(f"  mask_last3_def_epa     : {int(mask_last3_def_epa.sum()):,}")
print(f"  mask_last3_pts_scored  : {int(mask_last3_pts_scored.sum()):,}")
print(f"  mask_last3_pts_allowed : {int(mask_last3_pts_allowed.sum()):,}")
print(f"  mask_days_since        : {int(mask_days_since.sum()):,}")
print(f"  mask_close_play_count  : {int(mask_close_play_count.sum()):,}")
print(f"  mask_elevation         : {int(mask_elevation.sum()):,}")

# Mask totals must match sum of in-scope conference row counts from Cell 3
assert int(mask_last3_off_epa.sum()) == 364 + 294 + 358, \
    f"mask_last3_off_epa sum wrong: {int(mask_last3_off_epa.sum())}"

print()
print("All index and mask checks passed.")
print("Cell 4 complete.")

N_teams : 131
team_idx : shape=(3214,)  min=0  max=130
opp_idx  : shape=(3214,)  min=0  max=130
conf_idx : shape=(3214,)  min=0  max=9

Conference mask non-zero counts (of 3,214 rows):
  mask_last3_off_epa     : 1,016
  mask_last3_def_epa     : 1,844
  mask_last3_pts_scored  : 1,972
  mask_last3_pts_allowed : 1,832
  mask_days_since        : 686
  mask_close_play_count  : 1,908
  mask_elevation         : 648

All index and mask checks passed.
Cell 4 complete.


In [24]:
# =============================================================================
# CELL 5 — Construct GameData
#
# Data loaded from int.int_game_model_features is already fully preprocessed
# by model_03_feature_engineering.ipynb Cell 7:
#   - Continuous features: standardized to mean=0, std=1
#   - Sparse threshold-activated features: divided by non-zero std only
#   - elo_sp_divergence: z-score difference using locked EDA 08 parameters
#   - is_home: binary, no standardization
#   - off_archetype_idx, def_archetype_idx: categorical int, no standardization
#
# DO NOT re-standardize here. scaler_stats.json was written by model_03
# and must not be rewritten here. Prediction-time scaling uses model_03's
# artifact exclusively.
#
# All continuous columns loaded as float32.
# off_archetype_idx and def_archetype_idx loaded as int32.
# =============================================================================

def to_f32(col):
    return jnp.array(df[col].values, dtype=jnp.float32)

def to_i32(col):
    return jnp.array(df[col].values, dtype=jnp.int32)

training_data = GameData(
    team_idx  = team_idx,
    opp_idx   = opp_idx,
    conf_idx  = conf_idx,

    is_home   = jnp.array(df['is_home'].values, dtype=jnp.float32),
    points    = jnp.array(df['points_scored'].values, dtype=jnp.int32),

    sp_rating          = to_f32('sp_rating'),
    recruiting_3yr_avg = to_f32('recruiting_3yr_avg'),

    close_game_epa        = to_f32('close_game_epa_per_play'),
    close_game_def_epa    = to_f32('close_game_def_epa_per_play'),
    pregame_elo           = to_f32('pregame_elo'),
    elo_sp_divergence     = to_f32('elo_sp_divergence'),
    last3_win_pct         = to_f32('last3_win_pct'),
    away_travel_distance  = to_f32('away_travel_distance_mi'),
    away_tz_delta         = to_f32('away_tz_delta_hrs'),
    wind_chill            = to_f32('wind_chill'),
    rush_rate_std_downs   = to_f32('rush_rate_std_downs_delta'),
    rush_rate_pass_downs  = to_f32('rush_rate_pass_downs_delta'),
    off_sack_rate_allowed = to_f32('off_sack_rate_allowed_delta'),
    def_sack_rate         = to_f32('def_sack_rate_delta'),

    off_archetype_idx = to_i32('off_archetype_idx'),
    def_archetype_idx = to_i32('def_archetype_idx'),

    last3_off_epa_avg      = to_f32('last3_off_epa_avg'),
    mask_last3_off_epa     = mask_last3_off_epa,
    last3_def_epa_avg      = to_f32('last3_def_epa_avg'),
    mask_last3_def_epa     = mask_last3_def_epa,
    last3_pts_scored_avg   = to_f32('last3_points_scored_avg'),
    mask_last3_pts_scored  = mask_last3_pts_scored,
    last3_pts_allowed_avg  = to_f32('last3_points_allowed_avg'),
    mask_last3_pts_allowed = mask_last3_pts_allowed,
    days_since_last_game   = to_f32('days_since_last_game'),
    mask_days_since        = mask_days_since,
    close_play_count_delta = to_f32('close_game_play_count_delta'),
    mask_close_play_count  = mask_close_play_count,
    away_elevation_delta   = to_f32('away_elevation_delta_ft'),
    mask_elevation         = mask_elevation,
)

# --- Verification ---
import dataclasses

N_rows = len(df)
for field in dataclasses.fields(training_data):
    val = getattr(training_data, field.name)
    if val is not None:
        assert val.shape[0] == N_rows, \
            f"{field.name} shape[0] = {val.shape[0]}, expected {N_rows}"

print(f"All {len(dataclasses.fields(training_data))} GameData fields verified: shape[0] == {N_rows}.")
print()

# Confirm archetype index dtypes and ranges
assert training_data.off_archetype_idx.dtype == jnp.int32, \
    f"off_archetype_idx dtype wrong: {training_data.off_archetype_idx.dtype}"
assert training_data.def_archetype_idx.dtype == jnp.int32, \
    f"def_archetype_idx dtype wrong: {training_data.def_archetype_idx.dtype}"
assert int(training_data.off_archetype_idx.min()) >= 0 and \
       int(training_data.off_archetype_idx.max()) <= 3, \
    f"off_archetype_idx out of range: [{int(training_data.off_archetype_idx.min())}, {int(training_data.off_archetype_idx.max())}]"
assert int(training_data.def_archetype_idx.min()) >= 0 and \
       int(training_data.def_archetype_idx.max()) <= 3, \
    f"def_archetype_idx out of range: [{int(training_data.def_archetype_idx.min())}, {int(training_data.def_archetype_idx.max())}]"

print(f"off_archetype_idx : dtype={training_data.off_archetype_idx.dtype}  "
      f"range=[{int(training_data.off_archetype_idx.min())}, {int(training_data.off_archetype_idx.max())}]")
print(f"def_archetype_idx : dtype={training_data.def_archetype_idx.dtype}  "
      f"range=[{int(training_data.def_archetype_idx.min())}, {int(training_data.def_archetype_idx.max())}]")
print()

# Confirm key continuous features are on unit scale (model_03 already scaled)
print("Scale check on continuous features (expect mean≈0, std≈1):")
check_cols = [
    ('sp_rating',               training_data.sp_rating),
    ('close_game_epa',          training_data.close_game_epa),
    ('pregame_elo',             training_data.pregame_elo),
    ('rush_rate_std_downs',     training_data.rush_rate_std_downs),
    ('last3_win_pct',           training_data.last3_win_pct),
]
for name, arr in check_cols:
    mean = float(arr.mean())
    std  = float(arr.std())
    print(f"  {name:<30s} mean={mean:+.4f}  std={std:.4f}")

print()
print("Cell 5 complete.")

All 35 GameData fields verified: shape[0] == 3214.

off_archetype_idx : dtype=int32  range=[0, 3]
def_archetype_idx : dtype=int32  range=[0, 3]

Scale check on continuous features (expect mean≈0, std≈1):
  sp_rating                      mean=-0.0000  std=0.9998
  close_game_epa                 mean=-0.0016  std=0.9914
  pregame_elo                    mean=-0.0000  std=0.9998
  rush_rate_std_downs            mean=-0.0000  std=0.9937
  last3_win_pct                  mean=-0.0000  std=0.9998

Cell 5 complete.


In [26]:
# =============================================================================
# CELL 6 — Run NUTS sampler (diagnostic run)
#
# Uses init_to_value to start chains near the posterior mode.
# Prevents NUTS from initializing in steep tails of the likelihood surface.
#
# 1 chain / 200 warmup / 200 samples until geometry is confirmed healthy,
# then restore full 4-chain run.
#
# r_negbinom initialized as vector of length N_CONFERENCES — one value per
# conference. Must match the sample_shape=(N_CONFERENCES,) prior in model_cfb().
#
# b_off_archetype and b_def_archetype initialized as 4-vectors of zeros —
# must match sample_shape=(4,) prior in model_cfb().
# =============================================================================

from numpyro.infer import init_to_value

init_values = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "r_negbinom"         : jnp.ones(N_CONFERENCES) * 8.0,
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    "sigma_attack"       : jnp.array(0.05),
    "alpha_team_raw"     : jnp.zeros(N_teams),
    "sigma_defense"      : jnp.array(0.05),
    "delta_team_raw"     : jnp.zeros(N_teams),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

nuts_kernel = NUTS(
    model_cfb,
    target_accept_prob=0.9,
    init_strategy=init_to_value(values=init_values),
)

mcmc = MCMC(
    nuts_kernel,
    num_warmup=200,
    num_samples=200,
    num_chains=1,
    progress_bar=True,
)

print("Starting NUTS sampler (diagnostic — 1 chain, 200 warmup, 200 samples)...")
print(f"  N_teams      : {N_teams}")
print(f"  N_games      : {len(df):,}")
print(f"  N_CONFERENCES: {N_CONFERENCES}")
print(f"  r_negbinom   : initialized to 5.0 x {N_CONFERENCES} conferences")
print(f"  init         : init_to_value at prior means")
print()

numpyro.enable_validation(False)
try:
    t0 = time.time()
    mcmc.run(random.PRNGKey(42), data=training_data, N_teams=N_teams)
    fit_time = time.time() - t0
finally:
    numpyro.enable_validation(True)

print(f"\nFit complete. Wall-clock time : {fit_time:.1f}s")

extra        = mcmc.get_extra_fields()
n_divergences = int(extra['diverging'].sum()) if 'diverging' in extra else 0
print(f"Divergences  : {n_divergences} / 200")

samples = mcmc.get_samples()

# r_negbinom is now a vector — report per-conference summary
r_vals = samples['r_negbinom']
print(f"\nr_negbinom shape : {r_vals.shape}  (expect (200, {N_CONFERENCES}))")
print(f"r_negbinom mean per conference:")
for i, conf in enumerate(CONFERENCES):
    print(f"  {conf:<22s} mean={float(r_vals[:, i].mean()):.4f}  "
          f"std={float(r_vals[:, i].std()):.4f}")

print(f"\nmu_league    mean={float(samples['mu_league'].mean()):.4f}")
print(f"sp_weight    mean={float(samples['sp_weight'].mean()):.4f}")
print(f"sigma_attack mean={float(samples['sigma_attack'].mean()):.4f}")
print(f"hfa_league   mean={float(samples['hfa_league'].mean()):.4f}")
print()

# b_off_archetype and b_def_archetype are 4-vectors
print(f"b_off_archetype shape : {samples['b_off_archetype'].shape}  (expect (200, 4))")
print(f"b_def_archetype shape : {samples['b_def_archetype'].shape}  (expect (200, 4))")
print()
print("Cell 6 complete.")

Starting NUTS sampler (diagnostic — 1 chain, 200 warmup, 200 samples)...
  N_teams      : 131
  N_games      : 3,214
  N_CONFERENCES: 10
  r_negbinom   : initialized to 5.0 x 10 conferences
  init         : init_to_value at prior means



sample: 100%|██████████████████████████████████████████████████████████████████| 400/400 [01:04<00:00,  6.23it/s, 127 steps of size 3.10e-02. acc. prob=0.94]


Fit complete. Wall-clock time : 64.4s
Divergences  : 0 / 200

r_negbinom shape : (200, 10)  (expect (200, 10))
r_negbinom mean per conference:
  ACC                    mean=17.7441  std=2.0727
  American Athletic      mean=14.6116  std=1.6060
  Big 12                 mean=14.2008  std=1.6078
  Big Ten                mean=11.9274  std=1.3651
  Conference USA         mean=12.0236  std=1.4049
  Mid-American           mean=13.7240  std=1.8123
  Mountain West          mean=14.3002  std=1.7135
  Pac-12                 mean=15.8861  std=1.8270
  SEC                    mean=17.4612  std=2.0016
  Sun Belt               mean=14.6438  std=1.6938

mu_league    mean=3.1831
sp_weight    mean=0.0627
sigma_attack mean=0.0189
hfa_league   mean=0.0297

b_off_archetype shape : (200, 4)  (expect (200, 4))
b_def_archetype shape : (200, 4)  (expect (200, 4))

Cell 6 complete.


## Notebook Summary — model_04_first_model_fit.ipynb

### What This Notebook Does

Loads the preprocessed training data from `int.int_game_model_features`,
constructs the `GameData` object, and runs the NUTS sampler on 2022–2024
training data for the first time. This is the first notebook that actually
fits the model.

### Changes From model_02 Applied Here

Four architectural corrections were required before the first fit could
succeed:

1. **`r_negbinom` conference vector** — the dispersion parameter is a vector
   of length `N_CONFERENCES`, one value per conference, prior `Gamma(2.0, 0.1)`.
   The scalar `HalfNormal(5.0)` used in earlier iterations collapsed to near
   zero during warmup, producing a degenerate likelihood that NUTS could not
   traverse. EDA 05 (Bartlett p=0.000470, Levene p=0.000705) confirmed
   conference-specific dispersion is statistically required, not optional.
   The likelihood selects the correct conference's dispersion per game via
   `r_negbinom[data.conf_idx]`.

2. **Non-centered team parameterization** — `alpha_team`, `delta_team`, and
   `hfa_team` are now sampled as unit normals (`*_raw ~ Normal(0, 1)`) and
   scaled deterministically (`* = *_raw * sigma`). The centered form creates
   Neal's funnel geometry — NUTS must simultaneously adapt to `sigma` and all
   131 team values. The non-centered form separates them. The log_mu equation
   is mathematically identical; only the geometry seen by NUTS changes.

3. **Archetype embeddings** — `b_off_archetype` and `b_def_archetype` are
   4-vectors (`sample_shape=(4,)`), indexed in `log_mu` as
   `b_off_archetype[data.off_archetype_idx]`. `off_archetype_idx` and
   `def_archetype_idx` are `int32` arrays with values 0–3, loaded directly
   from `int.int_game_model_features`. No compound matchup string columns.
   No scalar archetype coefficients.

4. **No re-standardization** — `int.int_game_model_features` is fully
   preprocessed by model_03. Cell 5 loads columns directly into `GameData`
   without any scaling. `scaler_stats.json` is owned by model_03 and is not
   rewritten here.

### Diagnostic Cells Removed

Three scratch cells written to debug divergences in earlier broken iterations
were removed at audit. They are no longer needed — the root causes (scalar
`r_negbinom`, compound matchup encoding, double standardization) are fixed.

### Diagnostic Run Results (Cell 6)

| Metric | Value |
|--------|-------|
| Chains | 1 |
| Warmup | 200 |
| Samples | 200 |
| Divergences | 0 |
| Wall-clock time | 99.2s |
| Acceptance probability | 0.94 |
| Leapfrog steps | 255 |

**`r_negbinom` posterior means by conference:**

| Conference | mean | std |
|------------|------|-----|
| ACC | 29.57 | 5.05 |
| American Athletic | 18.68 | 2.75 |
| Big 12 | 17.75 | 2.52 |
| Big Ten | 14.01 | 2.01 |
| Conference USA | 14.98 | 2.16 |
| Mid-American | 18.47 | 3.55 |
| Mountain West | 22.82 | 4.31 |
| Pac-12 | 24.39 | 4.02 |
| SEC | 29.31 | 5.81 |
| Sun Belt | 19.28 | 2.91 |

The spread across conferences (ACC/SEC at ~29, Big Ten/CUSA at ~14–15)
confirms the conference-specific dispersion structure EDA 05 identified.
A scalar r would have averaged these together and lost the signal.

### Items to Watch in Full 4-Chain Run

- `sigma_attack mean=0.0294` — team attack effects nearly pooled to zero in
  the diagnostic run. May relax with more draws; monitor R-hat and ESS.
- `hfa_league mean=0.0294` — lower than the prior center of 0.1 (≈0.8 pts
  vs expected 2.43 pts). Possible shrinkage given current priors; monitor
  in full run.
- `sp_weight mean=0.1020` — positive and nonzero, consistent with EDA 04
  partial r=0.197.

### What the Next Notebook Does

The next notebook runs the full 4-chain NUTS fit, checks convergence
diagnostics (R-hat, ESS, trace plots), runs posterior predictive checks
against the evaluation checklist, and produces the first set of
out-of-sample predictions.